# 10 - Continuous Evaluation
## Shakespeare Evidence Desk

- **Goal:** measure the *assembled* end-to-end system — the frozen stage 05-08 pipeline with the stage 09 query-handling policy in front of it — on the labeled questions, replacing the lexical proxies used so far with a judged, defensible score, and turn those scores into explicit **release gates** that promote or roll back a build.
- **Experiment axis:** the evaluation **judge** — deterministic lexical proxies (control) vs. a single Large Language Model (LLM) correctness judge vs. a faceted judge scoring the Retrieval-Augmented Generation (RAG) triad of faithfulness, answer relevance, and context relevance (the pipeline is held fixed; only *how we score it* changes).
- **Loads from Steps 05-09:** every stage handoff plus `.artifacts/query-handoff.json` (the selected query policy) and the labeled `eval-questions.json`, replayed to produce one answer per question through the exact production path.
- **Exports:** `.artifacts/evaluation-results.json` — the chosen judge, per-category and triad scores on the development and held-out splits, retrieval metrics, and the promotion/rollback gates stages 11-12 enforce before shipping.


## Why This Stage

*In general — what continuous evaluation is and why a serious Retrieval-Augmented Generation (RAG) system cannot ship without it.* Every stage so far ended with a measurement, but each measured only its own slice against a cheap proxy. No stage has yet asked the question a user actually cares about: *given a real question, is the final answer faithful to the evidence, relevant to the ask, and built on context that was actually about the question?* This stage assembles the whole pipeline — the stage 09 query policy feeding the stage 05-08 retrieval, reranking, context, and grounding stack — and scores its end-to-end answers with a validated judge instead of a word-overlap heuristic.

The deeper purpose is *continuity*. A RAG system is never finished: the corpus grows, prompts get tuned, models get swapped, retrieval parameters drift. Any of those changes can silently break the behavior the pipeline exists to guarantee — grounded, cited, abstaining answers. Continuous evaluation is the safety rail that makes change safe: a fixed labeled set, a judge calibrated against the labels, and explicit **gates** that a build must clear to ship. With the gates in place, "did this change help or hurt?" stops being a matter of opinion and becomes a number on the held-out split.

The choice here is *measured*, like every other stage, but what we are choosing is different: not a pipeline component, but the **evaluation method** itself. More judging means more cost and a model grading a model, so we keep the simplest judge whose verdicts agree with the ground truth the labels give us, add the faceted triad only where it localizes a fault the labels cannot see, and keep the free lexical proxy as the gate that runs on every commit. The output is not a better answer — it is the standing definition of "good enough to ship."


## Conceptual Foundations

### Evaluation is the stage that makes every earlier decision honest

Stages 05-09 each picked an option by measuring it, but always against *lexical proxies* — word overlap, work-name matching, a refusal sentinel. Those proxies are cheap and deterministic, and they catch blatant failures, but they are blind to paraphrase and cannot tell whether a citation actually *supports* the claim next to it. Continuous evaluation replaces the proxy with a defensible judgment and, crucially, turns a one-off measurement into a **standing gate**: every future change to any stage must clear the same bar before it ships.

### The Retrieval-Augmented Generation (RAG) triad

A RAG answer can fail in two different places, and a single score hides which. The **triad** separates them:

- **Faithfulness** — is every claim in the answer supported by the retrieved context? This isolates *generation* faults (the model drifting from the evidence).
- **Answer relevance** — does the answer actually address the question that was asked? This catches fluent answers to the wrong question.
- **Context relevance** — is the retrieved context relevant to the question? This isolates *retrieval* faults (the evidence never arrived), independent of what the generator did with it.

Reading the three together tells you *where* to fix a regression: low context relevance points upstream to retrieval or query handling; low faithfulness with high context relevance points at the generator.

### Offline evaluation, development and held-out splits

**Offline evaluation** scores the system on a fixed labeled set before release, as opposed to **online evaluation**, which samples real production traffic (stage 12's concern). The labeled set is split: the **development** split is where every earlier stage tuned its choice, so scoring well on it partly reflects that tuning. The **held-out** split was never used for tuning, so it is the honest estimate of generalization — the gates are read from the held-out split for exactly this reason.

### Judges, and why a judge must itself be validated

An LLM judge is a model scoring a model, so it can be wrong or biased. Before trusting it we **validate** it against the ground truth we actually have: the labels tell us which questions are unanswerable (the system must abstain) and which works each answer should reference. A judge whose verdicts contradict those labels cannot be trusted on the softer axes. We measure that agreement and only then lean on the judge for faithfulness and relevance, which the labels cannot capture.

### Release gates: promotion and rollback

A **gate** is a threshold a build must clear to ship. Some gates are **hard constraints** inherited from earlier stages — perfect abstention on out-of-corpus questions, every answer cited — and any build that breaks one is rolled back regardless of its other scores. The rest are quality thresholds (faithfulness, answer relevance, context relevance, retrieval recall). **Promotion** means all gates pass on the held-out split; **rollback** means reverting to the last build that did.

### What evaluation is *not* (clean stage boundaries)

- **Not a pipeline stage:** it changes nothing about how answers are produced; it scores the frozen system and decides whether it may ship.
- **Not query handling (09) or generation (08):** it never rewrites a query or writes an answer; it grades the answers those stages produced.
- **Not operations (12):** it runs offline on a labeled set; live monitoring, adversarial drills, and alerting on real traffic are stage 12.

### Common misconceptions and failure modes

- **"A high average score means the system is good."** An average hides the category that fails. A build can post a strong mean while silently breaking the cross-play questions; per-category scores and hard gates exist to catch that.
- **"The LLM judge is ground truth."** It is an instrument that must be calibrated. An unvalidated judge can agree with a confident, wrong answer; that is why we check it against the labels first and keep the cheap lexical gate alongside it.
- **"Tune until the held-out score is perfect."** Tuning against the held-out split destroys the only honest generalization estimate you have. The held-out split is read, never optimized against.
- **"More metrics are always better."** Metrics only help if each one changes a decision. The triad earns its place because each facet localizes a different fault; a metric that never moves a gate is noise.


## The Technique Landscape

We **fix** the whole pipeline — the stage 09 query policy in front of the stage 05-08 retrieval, reranking, context, and grounding stack — and vary only the **judge** that scores its answers. Three judges are compared head-to-head; the rest are explained so the choice is informed.

| Judge / method | Role here | What it does | Strengths | Weaknesses |
|---|---|---|---|---|
| **Lexical proxies** | compared (control) | word-overlap grounding, work-name matching, sentinel-based abstention | free, deterministic, fast; good Continuous Integration (CI) pre-filter | blind to paraphrase and to whether a citation truly supports a claim |
| **Single LLM judge** | compared | one call per answer returning a correct / incorrect verdict with a reason | cheap judgment that captures meaning lexical misses | coarse; one bit hides *why* an answer failed |
| **Faceted LLM judge (RAG triad)** | compared | one call scoring faithfulness, answer relevance, and context relevance each on 0-1 | separates retrieval faults from generation faults; the signal the gates need | more tokens; judge can be biased or miscalibrated |
| **Human evaluation** | explained, not run | expert graders score answers against gold evidence | the gold standard for quality | not scalable or repeatable in a notebook; reserved for periodic audits |
| **Embedding similarity to a gold answer** | explained, not run | cosine similarity between the answer and a reference answer | cheap and deterministic | needs written gold answers we do not have; rewards wording over truth |
| **Pairwise / Elo ranking** | explained, not run | judge compares two systems' answers head-to-head | strong for ranking model variants | needs two systems and many comparisons; overkill for one build |

So three judges are measured; the rest frame the trade-off. The lexical proxies are kept even after selection, as the cheap gate that runs on every commit.


## 1. Load the Previous Selection

- **Purpose:** evaluation scores the *assembled* system, so this section rebuilds the entire production path from the handoffs — the stage 05-08 retrieval, reranking, context builder, and grounding contract, plus the stage 09 query-handling policy in front of them — and produces one answer per labeled question through exactly that path. Nothing here is re-decided; it is the first time every earlier selection runs together.
- **What the code does:** locates the repository root; reads every upstream handoff (`query-handoff.json`, `generation-handoff.json`, `context-handoff.json`, and the retrieval, reranking, index, and chunk artifacts); rebuilds Best Matching 25 (BM25), dense, and hybrid Reciprocal Rank Fusion (RRF) retrieval, the selected reranker and context builder, the cached embedder, the strict generator, and the stage 09 query operations (rewrite, decompose, route); then runs the **selected** query policy end to end over both the development and held-out splits, caching every model call to disk.
- **Fails loudly, early:** if `query-handoff.json` is missing, run notebook 09 first; if any earlier artifact is missing, run notebooks 04-08. If the index was built on offline-fallback vectors (`embedding_backend != 'hosted'`), the notebook **stops here** — evaluating fake retrieval certifies nothing.
- **Expected result:** the frozen pipeline configuration, the selected query policy, the Azure deployments, and how many development and held-out questions were answered. Reformulations, embeddings, and generations are served from the stage 03-09 disk caches, so a warm run bills nothing.


In [ ]:
import hashlib
import json
import math
import re
import time
from collections import defaultdict
from pathlib import Path

import tiktoken
from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
query_handoff_path = ARTIFACTS / 'query-handoff.json'
generation_handoff_path = ARTIFACTS / 'generation-handoff.json'
context_handoff_path = ARTIFACTS / 'context-handoff.json'
index_path = ARTIFACTS / 'index.json'
retrieval_handoff_path = ARTIFACTS / 'retrieval-handoff.json'
reranking_handoff_path = ARTIFACTS / 'reranking-handoff.json'
chunks_path = ARTIFACTS / 'chunks.json'
questions_path = ARTIFACTS / 'eval-questions.json'
if not query_handoff_path.is_file():
    raise RuntimeError('Missing .artifacts/query-handoff.json. Run notebook 09 first.')
for required in (generation_handoff_path, context_handoff_path, index_path, retrieval_handoff_path, reranking_handoff_path):
    if not required.is_file():
        raise RuntimeError(f'Missing .artifacts/{required.name}. Run notebooks 04-08 first.')

query_handoff = json.loads(query_handoff_path.read_text(encoding='utf-8'))
generation_handoff = json.loads(generation_handoff_path.read_text(encoding='utf-8'))
context_handoff = json.loads(context_handoff_path.read_text(encoding='utf-8'))
index_payload = json.loads(index_path.read_text(encoding='utf-8'))
retrieval_handoff = json.loads(retrieval_handoff_path.read_text(encoding='utf-8'))
reranking_handoff = json.loads(reranking_handoff_path.read_text(encoding='utf-8'))
chunks_payload = json.loads(chunks_path.read_text(encoding='utf-8')) if chunks_path.is_file() else {'chunks': []}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

index_type = index_payload['index_type']
stored_dims = index_payload['dims']
embedding_backend = index_payload.get('embedding_backend', 'unknown')
records = index_payload['records']
if embedding_backend != 'hosted':
    raise RuntimeError(
        f"index.json was built on the '{embedding_backend}' backend, not hosted Azure. "
        "Re-run notebooks 03-09 against Azure before evaluating.")

# ---- frozen upstream decisions (never re-decided here) ----
SELECTED_METHOD = retrieval_handoff['selected_method']
RRF_K = retrieval_handoff.get('rrf_k') or 60
bm25_params = retrieval_handoff.get('bm25_params', {'k1': 1.5, 'b': 0.75})
SELECTED_RERANKER = reranking_handoff['selected_reranker']
TOP_K = reranking_handoff.get('top_k', 5)
SELECTED_BUILDER = context_handoff['selected_builder']
CONTEXT_TOKEN_BUDGET = context_handoff['token_budget']
CONTEXT_POOL = context_handoff.get('candidate_pool', reranking_handoff.get('rerank_pool', 20))
DEDUP_THRESHOLD = context_handoff.get('dedup', {}).get('threshold', 0.6)
PARENT_NEIGHBORS = context_handoff.get('parent_context', {}).get('neighbors_each_side', 1)
PARENT_MAX_TOKENS = context_handoff.get('parent_context', {}).get('max_tokens', 700)
SELECTED_CONTRACT = generation_handoff['selected_contract']
GEN_SYSTEM_PROMPT = generation_handoff['system_prompt']
GEN_USES_CONTEXT = generation_handoff.get('uses_context', True)
MAX_COMPLETION_TOKENS = generation_handoff.get('max_completion_tokens', 2000)
ABSTAIN_SENTINEL = generation_handoff.get('abstention', {}).get('sentinel', 'INSUFFICIENT_CONTEXT')

# ---- the stage 09 query-handling policy this stage evaluates ----
SELECTED_POLICY = query_handoff['selected_policy']
policy_params = query_handoff.get('policy_parameters', {})
MAX_SUBQUERIES = policy_params.get('max_subqueries', 3)
RETRY_POOL_MULTIPLIER = policy_params.get('retry_pool_multiplier', 2)
REWRITE_SYSTEM = query_handoff['prompts']['rewrite_system']
DECOMPOSE_SYSTEM = query_handoff['prompts']['decompose_system']
ROUTE_SYSTEM = query_handoff['prompts']['route_system']

vectors = [r['embedding'] for r in records]
N = len(vectors)
work_titles = {r['work_id']: r['work'] for r in records}
encoder = tiktoken.get_encoding('cl100k_base')


def count_tokens(text):
    return len(encoder.encode(text))


def dot(a, b):
    return sum(x * y for x, y in zip(a, b))


def l2_normalize(vector):
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector


_WORD_PATTERN = re.compile(r"[a-z0-9']+")


def tokenize(text):
    return _WORD_PATTERN.findall(text.lower())


def exact_search_rows(query_vector, candidate_rows=None):
    pool = range(N) if candidate_rows is None else candidate_rows
    return sorted(pool, key=lambda i: dot(query_vector, vectors[i]), reverse=True)


if index_type == 'ivf':
    ivf = index_payload['ivf']
    ivf_centroids = ivf['centroids']
    ivf_lists = {int(cluster): rows for cluster, rows in ivf['lists'].items()}
    nprobe = index_payload['params']['nprobe']

    def dense_search(query_vector, k, candidate_rows=None):
        nearest = sorted(range(len(ivf_centroids)),
                         key=lambda c: dot(query_vector, ivf_centroids[c]), reverse=True)[:nprobe]
        pool = [i for c in nearest for i in ivf_lists.get(c, ())]
        if candidate_rows is not None:
            allowed = set(candidate_rows)
            pool = [i for i in pool if i in allowed]
        return exact_search_rows(query_vector, pool)[:k]
else:
    def dense_search(query_vector, k, candidate_rows=None):
        return exact_search_rows(query_vector, candidate_rows)[:k]


class BM25:
    """The transparent Okapi BM25 keyword index from notebook 05, reused to replay its pool."""

    def __init__(self, documents, term_saturation=1.5, length_strength=0.75):
        self.k1 = term_saturation
        self.b = length_strength
        self.tokenized_docs = [tokenize(text) for text in documents]
        self.doc_lengths = [len(tokens) for tokens in self.tokenized_docs]
        self.avg_len = (sum(self.doc_lengths) / len(self.tokenized_docs)) if self.tokenized_docs else 0.0
        self.term_counts_per_doc = []
        document_frequency = defaultdict(int)
        for tokens in self.tokenized_docs:
            term_counts = defaultdict(int)
            for term in tokens:
                term_counts[term] += 1
            self.term_counts_per_doc.append(term_counts)
            for term in term_counts:
                document_frequency[term] += 1
        total_documents = len(self.tokenized_docs)
        self.idf = {
            term: math.log(1 + (total_documents - doc_count + 0.5) / (doc_count + 0.5))
            for term, doc_count in document_frequency.items()
        }

    def search(self, query, top_k, candidate_rows=None):
        query_terms = [term for term in tokenize(query) if term in self.idf]
        candidate_indices = range(len(self.tokenized_docs)) if candidate_rows is None else candidate_rows
        scored_rows = []
        for row_index in candidate_indices:
            term_counts = self.term_counts_per_doc[row_index]
            doc_length = self.doc_lengths[row_index]
            score = 0.0
            for term in query_terms:
                term_count = term_counts.get(term, 0)
                if not term_count:
                    continue
                length_normalizer = self.k1 * (1 - self.b + self.b * doc_length / (self.avg_len or 1))
                score += self.idf[term] * (term_count * (self.k1 + 1)) / (term_count + length_normalizer)
            if score > 0:
                scored_rows.append((row_index, score))
        scored_rows.sort(key=lambda row_and_score: row_and_score[1], reverse=True)
        return [row_index for row_index, _ in scored_rows[:top_k]]


bm25 = BM25([record['text'] for record in records],
            term_saturation=bm25_params.get('k1', 1.5), length_strength=bm25_params.get('b', 0.75))


def reciprocal_rank_fusion(rankings, damping, top_k):
    fused_scores = defaultdict(float)
    for ranking in rankings:
        for rank, row_index in enumerate(ranking):
            fused_scores[row_index] += 1.0 / (damping + rank + 1)
    ranked_rows = sorted(fused_scores.items(), key=lambda row_and_score: row_and_score[1], reverse=True)
    return [row_index for row_index, _ in ranked_rows[:top_k]]


def retrieve_pool(query_text, query_vector, pool_size):
    if SELECTED_METHOD == 'bm25':
        return bm25.search(query_text, pool_size)
    if SELECTED_METHOD == 'dense':
        return dense_search(query_vector, pool_size)
    dense_ranking = dense_search(query_vector, pool_size)
    keyword_ranking = bm25.search(query_text, pool_size)
    return reciprocal_rank_fusion([dense_ranking, keyword_ranking], RRF_K, pool_size)


chunks_by_id = {c['chunk_id']: c for c in chunks_payload.get('chunks', [])}


def chunk_index(chunk_id):
    tail = chunk_id.rsplit(':', 1)[-1]
    return int(tail) if tail.isdigit() else None


siblings_by_work = defaultdict(dict)
for chunk_id, chunk in chunks_by_id.items():
    position = chunk_index(chunk_id)
    if position is not None:
        siblings_by_work[chunk.get('work_id')][position] = chunk_id

config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)
embedding_deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']
chat_deployment = config['AZURE_OPENAI_CHAT_DEPLOYMENT']

EMBED_CACHE = ARTIFACTS / 'embedding-cache'
GEN_CACHE = ARTIFACTS / 'generation-cache'
QUERY_CACHE = ARTIFACTS / 'query-cache'
for cache_dir in (EMBED_CACHE, GEN_CACHE, QUERY_CACHE):
    cache_dir.mkdir(parents=True, exist_ok=True)
live_calls = {'embed': 0, 'query': 0, 'generation': 0, 'judge': 0}


def embed_query(text):
    cache_file = EMBED_CACHE / (hashlib.sha256(f'{embedding_deployment}|{text}'.encode('utf-8')).hexdigest() + '.json')
    if cache_file.is_file():
        full_vector = json.loads(cache_file.read_text())
    else:
        full_vector = client.embeddings.create(model=embedding_deployment, input=[text]).data[0].embedding
        cache_file.write_text(json.dumps(full_vector))
        live_calls['embed'] += 1
    return l2_normalize(full_vector[:stored_dims])


def apply_selected_reranker(query_text, query_vector, candidate_rows):
    if SELECTED_RERANKER == 'semantic':
        return sorted(candidate_rows, key=lambda row: dot(query_vector, vectors[row]), reverse=True)
    return list(candidate_rows)


def token_set(text):
    return set(tokenize(text))


def jaccard(a, b):
    union = len(a | b)
    return len(a & b) / union if union else 0.0


def format_scene(scenes):
    if not scenes:
        return 'Unknown scene'
    act, scene = scenes[0]
    return f'Act {act}, Scene {scene}'


def parent_text(row):
    record = records[row]
    work_id = record['work_id']
    position = chunk_index(record['chunk_id'])
    if position is None or work_id not in siblings_by_work:
        return record['text']
    pieces = []
    for offset in range(-PARENT_NEIGHBORS, PARENT_NEIGHBORS + 1):
        sibling_id = siblings_by_work[work_id].get(position + offset)
        sibling = chunks_by_id.get(sibling_id) if sibling_id else None
        if sibling and any(scene in record['scenes'] for scene in sibling.get('scenes', [])):
            pieces.append((position + offset, sibling['text']))
    pieces.sort()
    seen_lines, merged_lines = set(), []
    for _, text in pieces:
        for line in text.split('\n'):
            stripped = line.strip()
            if stripped and stripped in seen_lines:
                continue
            if stripped:
                seen_lines.add(stripped)
            merged_lines.append(line)
    merged = '\n'.join(merged_lines).strip() or record['text']
    token_ids = encoder.encode(merged)
    return encoder.decode(token_ids[:PARENT_MAX_TOKENS]) if len(token_ids) > PARENT_MAX_TOKENS else merged


def passage_block(position, record, text):
    header = f"[{position}] {record['work']} - {format_scene(record['scenes'])}"
    if record['speakers']:
        header += f" ({', '.join(record['speakers'][:4])})"
    return f"{header}\n{text.strip()}"


def assemble(rows, texts):
    blocks, citations = [], []
    for position, (row, text) in enumerate(zip(rows, texts), start=1):
        record = records[row]
        blocks.append(passage_block(position, record, text))
        citations.append({
            'marker': position, 'chunk_id': record['chunk_id'], 'work': record['work'],
            'work_id': record['work_id'], 'scene': format_scene(record['scenes']), 'speakers': record['speakers'],
        })
    return {'rows': rows, 'context': '\n\n'.join(blocks), 'citations': citations}


def build_raw(ordered_rows):
    chosen = list(ordered_rows[:TOP_K])
    return assemble(chosen, [records[row]['text'] for row in chosen])


def _pack(ordered_rows, expand):
    kept, kept_signatures, texts, used_tokens = [], [], [], 0
    for row in ordered_rows:
        signature = token_set(records[row]['text'])
        if any(jaccard(signature, previous) >= DEDUP_THRESHOLD for previous in kept_signatures):
            continue
        text = parent_text(row) if expand else records[row]['text']
        block_tokens = count_tokens(passage_block(len(kept) + 1, records[row], text)) + 2
        if used_tokens + block_tokens > CONTEXT_TOKEN_BUDGET:
            continue
        kept.append(row)
        kept_signatures.append(signature)
        texts.append(text)
        used_tokens += block_tokens
    return assemble(kept, texts)


BUILDERS = {'raw': build_raw, 'dedup-budget': lambda r: _pack(r, False), 'dedup-parent': lambda r: _pack(r, True)}
build_context = BUILDERS[SELECTED_BUILDER]


def build_context_for_queries(primary_query, variants, pool_size):
    rankings = []
    for variant in variants:
        rankings.append(retrieve_pool(variant, embed_query(variant), pool_size))
    fused = rankings[0] if len(rankings) == 1 else reciprocal_rank_fusion(rankings, RRF_K, pool_size)
    ordered = apply_selected_reranker(primary_query, embed_query(primary_query), fused)
    return build_context(ordered)


def generate(question_text, context_text):
    user_message = (f'Context:\n{context_text}\n\nQuestion: {question_text}'
                    if GEN_USES_CONTEXT else f'Question: {question_text}')
    cache_key = hashlib.sha256(
        f"{chat_deployment}|{SELECTED_CONTRACT}|{MAX_COMPLETION_TOKENS}|{user_message}".encode('utf-8')).hexdigest()
    cache_file = GEN_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        cached = json.loads(cache_file.read_text())
        return cached['answer'], cached.get('latency_ms', 0.0)
    started = time.perf_counter()
    response = client.chat.completions.create(
        model=chat_deployment, max_completion_tokens=MAX_COMPLETION_TOKENS,
        messages=[{'role': 'system', 'content': GEN_SYSTEM_PROMPT}, {'role': 'user', 'content': user_message}])
    latency_ms = (time.perf_counter() - started) * 1000
    answer = (response.choices[0].message.content or '').strip()
    cache_file.write_text(json.dumps({'answer': answer, 'latency_ms': latency_ms}))
    live_calls['generation'] += 1
    return answer, latency_ms


def _query_chat_json(system_prompt, user_text, operation):
    cache_key = hashlib.sha256(f'{chat_deployment}|{operation}|{user_text}'.encode('utf-8')).hexdigest()
    cache_file = QUERY_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        return json.loads(cache_file.read_text())
    response = client.chat.completions.create(
        model=chat_deployment, response_format={'type': 'json_object'},
        messages=[{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_text}])
    parsed = json.loads(re.search(r'\{.*\}', response.choices[0].message.content or '{}', re.DOTALL).group(0))
    cache_file.write_text(json.dumps(parsed))
    live_calls['query'] += 1
    return parsed


def rewrite_query(q):
    return (_query_chat_json(REWRITE_SYSTEM, q, 'rewrite').get('query') or '').strip() or q


def decompose_query(q):
    subs = [str(s).strip() for s in _query_chat_json(DECOMPOSE_SYSTEM, q, 'decompose').get('subqueries', []) if str(s).strip()]
    return subs[:MAX_SUBQUERIES] or [q]


def route_query(q):
    out = _query_chat_json(ROUTE_SYSTEM, q, 'route')
    complexity = out.get('complexity') if out.get('complexity') in ('simple', 'multi') else 'simple'
    return {'complexity': complexity, 'likely_in_corpus': bool(out.get('likely_in_corpus', True))}


def is_abstention(answer):
    return ABSTAIN_SENTINEL in answer.upper()


def answer_question(question_text):
    """Run the stage 09 selected policy end to end: shape the query, retrieve, build context, generate."""
    route = None
    if SELECTED_POLICY == 'passthrough':
        primary, variants = question_text, [question_text]
    elif SELECTED_POLICY == 'rewrite':
        primary = rewrite_query(question_text); variants = [primary]
    elif SELECTED_POLICY == 'decompose':
        variants = decompose_query(question_text); primary = variants[0]
    else:  # adaptive
        route = route_query(question_text)
        if route['complexity'] == 'multi':
            variants = decompose_query(question_text); primary = variants[0]
        else:
            primary = rewrite_query(question_text); variants = [primary]
    built = build_context_for_queries(primary, variants, CONTEXT_POOL)
    answer, latency = generate(question_text, built['context'])
    if SELECTED_POLICY == 'adaptive' and is_abstention(answer) and route['likely_in_corpus']:
        variants = list(dict.fromkeys(variants + [rewrite_query(question_text)]))
        built = build_context_for_queries(primary, variants, CONTEXT_POOL * RETRY_POOL_MULTIPLIER)
        answer, retry_latency = generate(question_text, built['context'])
        latency += retry_latency
    return {'context': built['context'], 'citations': built['citations'], 'answer': answer, 'latency_ms': latency}


# ---- run the production path once per labeled question, on BOTH splits ----
def run_split(split_questions):
    runs = []
    for question in split_questions:
        produced = answer_question(question['question'])
        produced.update({
            'id': question.get('id'), 'question': question['question'], 'category': question.get('category', 'uncategorized'),
            'answerable': bool(question.get('answerable')), 'expected_works': question.get('expected_works', []),
            'expected_evidence': question.get('expected_evidence', ''),
        })
        runs.append(produced)
    return runs


dev_runs = run_split(evaluation_questions.get('development', []))
heldout_runs = run_split(evaluation_questions.get('heldout', []))

print(f'Frozen pipeline: query-policy={SELECTED_POLICY}, retrieval={SELECTED_METHOD}, reranker={SELECTED_RERANKER}, '
      f'builder={SELECTED_BUILDER}, contract={SELECTED_CONTRACT}, top-k={TOP_K}.')
print(f'Azure deployments: chat={chat_deployment}, embedding={embedding_deployment}.')
print(f'Answered {len(dev_runs)} development and {len(heldout_runs)} held-out question(s) through the production path.')
print(f'Live calls this run: {live_calls} (rest served from the stage 03-09 disk caches).')


## 2. Explain the Alternatives

A judge is **good** when its verdicts agree with the ground truth we can check (unanswerable questions must be abstained, answers should name the expected works), when it adds signal the cheap proxies cannot (faithfulness, answer relevance, context relevance), and when it does so at a cost worth paying. We **fix the pipeline** — every answer in Section 1 is already produced — and vary only the **judge** that scores those answers.

### What we actually compare vs. what we only explain

| Approach | Role here | Why |
|---|---|---|
| **Lexical proxies** | compared (control) | the deterministic heuristics every earlier stage used; the baseline a judge must beat to justify its cost, and the gate cheap enough to run on every commit |
| **Single Large Language Model (LLM) judge** | compared | one call returns a correct / incorrect verdict; tests whether a coarse semantic judgment already agrees with the labels better than word overlap |
| **Faceted LLM judge (Retrieval-Augmented Generation (RAG) triad)** | compared | one call scores faithfulness, answer relevance, and context relevance; tests whether separating retrieval faults from generation faults is worth the extra tokens |
| **Human evaluation** | explained, not run | the gold standard, but unrepeatable in a notebook and unsuitable as a per-commit gate; reserved for periodic audits of the judge itself |
| **Embedding similarity to a gold answer** | explained, not run | needs written reference answers the corpus labels do not provide, and rewards matching wording over matching truth |
| **Pairwise / Elo ranking** | explained, not run | compares two systems rather than scoring one build against an absolute bar; a model-selection tool, not a release gate |

So three judges are compared head-to-head; the rest are explained so the choice makes sense.

### How each compared judge actually works (a line or two)

- **Lexical proxies** — reuse the stage 08/09 heuristics: grounding is the share of an answer's content words found in its context, abstention is sentinel detection, and work relevance is name matching. Zero model calls, fully deterministic, blind to paraphrase.
- **Single LLM judge** — one chat call per answer sees the question, the context, and the answer (never the label) and returns `{"correct": true|false, "reason": "..."}`. It captures meaning the proxies miss but compresses every failure mode into one bit.
- **Faceted LLM judge** — one chat call returns the triad: faithfulness, answer relevance, and context relevance, each on 0-1. The three facets localize *where* an answer failed, which is exactly what a regression investigation needs.

### Why this is the experiment we run

The stage exists to make "is it good enough to ship?" a defensible number. That number is only trustworthy if the judge producing it is itself trustworthy, so the experiment is really judge *validation*: we measure how well each judge's verdicts agree with the label-derived ground truth (abstention correctness and work relevance), keep the cheapest judge that agrees, and add the faceted triad because faithfulness and context relevance — the two axes the gates most depend on — are precisely what the labels cannot measure. The thresholds and the dev/held-out split are settings applied after the judge is chosen, not competing arms.


## 3. Run Controlled Experiments

- **Fixed pipeline:** the answers were produced once in Section 1 through the production path; all three judges score the *same* answers, so any difference is the judge, not the system.
- **What the code does:** defines the three judges — the deterministic **lexical** proxy, the **single** LLM correctness judge, and the **faceted** RAG-triad judge — with the two LLM judges cached to disk by (deployment, judge, question, answer); runs every judge over every development answer; and separately records the label-derived ground truth (did the system abstain exactly on the unanswerable questions, and do answers name the expected works) so Section 4 can measure judge-label agreement.
- **The judge never sees the label.** It scores the question, context, and answer only. This keeps the agreement measurement honest: a judge that agreed because it was told the answer would prove nothing.
- **LLM calls:** each judged answer is one cached chat completion from the reasoning model; a warm cache makes zero live calls, and the lexical judge makes none ever.
- **Expected result:** the three registered judges, the number of verdicts each produced, the live-vs-cached call counts, and confirmation that all judges scored the identical Section 1 answers.


In [ ]:
_STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'of', 'to', 'in', 'on', 'for', 'with', 'as', 'by', 'at',
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'it', 'its', 'this', 'that', 'these', 'those',
    'he', 'she', 'they', 'his', 'her', 'their', 'who', 'what', 'which', 'how', 'does', 'do', 'did',
    'from', 'not', 'no', 'has', 'have', 'had', 'about', 'into', 'than', 'then', 'there', 'here',
}
_CITATION_PATTERN = re.compile(r'\[(\d+)\]')


def grounding_overlap(answer, context_text):
    answer_terms = {t for t in tokenize(_CITATION_PATTERN.sub(' ', answer)) if t not in _STOPWORDS}
    if not answer_terms:
        return 0.0
    return len(answer_terms & set(tokenize(context_text))) / len(answer_terms)


def names_expected_works(answer, expected_works):
    lowered = answer.lower()
    return all(str(w).lower() in lowered for w in expected_works) if expected_works else True


# ---- Judge 1: deterministic lexical proxy (zero model calls) ----
LEXICAL_GROUNDING_THRESHOLD = 0.35


def judge_lexical(run):
    if not run['answerable']:
        return {'correct': is_abstention(run['answer']), 'faithfulness': None,
                'answer_relevance': None, 'context_relevance': None}
    if is_abstention(run['answer']):
        return {'correct': False, 'faithfulness': 0.0, 'answer_relevance': 0.0, 'context_relevance': None}
    grounded = grounding_overlap(run['answer'], run['context'])
    correct = grounded >= LEXICAL_GROUNDING_THRESHOLD and names_expected_works(run['answer'], run['expected_works'])
    return {'correct': correct, 'faithfulness': grounded,
            'answer_relevance': 1.0 if names_expected_works(run['answer'], run['expected_works']) else 0.0,
            'context_relevance': None}


EVAL_CACHE = ARTIFACTS / 'eval-cache'
EVAL_CACHE.mkdir(parents=True, exist_ok=True)


def _judge_chat_json(judge_name, system_prompt, user_text):
    cache_key = hashlib.sha256(f'{chat_deployment}|{judge_name}|{user_text}'.encode('utf-8')).hexdigest()
    cache_file = EVAL_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        return json.loads(cache_file.read_text())
    response = client.chat.completions.create(
        model=chat_deployment, response_format={'type': 'json_object'},
        messages=[{'role': 'system', 'content': system_prompt}, {'role': 'user', 'content': user_text}])
    parsed = json.loads(re.search(r'\{.*\}', response.choices[0].message.content or '{}', re.DOTALL).group(0))
    cache_file.write_text(json.dumps(parsed))
    live_calls['judge'] += 1
    return parsed


# ---- Judge 2: single LLM correctness judge ----
SINGLE_JUDGE_SYSTEM = (
    'You grade a Shakespeare evidence desk. You see a question, the numbered context passages the system '
    'retrieved, and its answer. Decide whether the answer correctly and faithfully answers the question using '
    'ONLY the context. An answer that states facts the context does not support, or that fails to address the '
    'question, is incorrect. The refusal token means the system declined to answer. Do not use outside knowledge '
    'about Shakespeare. Return a JSON object {"correct": true|false, "reason": "..."}.'
)


def judge_single(run):
    user_text = f"Question: {run['question']}\n\nContext:\n{run['context']}\n\nAnswer: {run['answer']}"
    out = _judge_chat_json('single', SINGLE_JUDGE_SYSTEM, user_text)
    return {'correct': bool(out.get('correct')), 'faithfulness': 1.0 if out.get('correct') else 0.0,
            'answer_relevance': 1.0 if out.get('correct') else 0.0, 'context_relevance': None,
            'reason': out.get('reason', '')}


# ---- Judge 3: faceted RAG-triad judge ----
FACETED_JUDGE_SYSTEM = (
    'You grade a Shakespeare evidence desk on three independent axes, each from 0.0 to 1.0. You see a question, '
    'the numbered context passages, and the answer.\n'
    '- faithfulness: fraction of the answer supported by the context (1.0 = every claim grounded, 0.0 = unsupported).\n'
    '- answer_relevance: how directly the answer addresses the question (1.0 = fully, 0.0 = off-topic).\n'
    '- context_relevance: how relevant the retrieved context is to the question (1.0 = on point, 0.0 = unrelated).\n'
    'Judge only the text shown; do not use outside knowledge. The refusal token is a non-answer: score faithfulness '
    'and answer_relevance 0.0 but still score context_relevance. '
    'Return a JSON object {"faithfulness": x, "answer_relevance": x, "context_relevance": x}.'
)


def _clip01(value):
    try:
        return max(0.0, min(1.0, float(value)))
    except (TypeError, ValueError):
        return 0.0


def judge_faceted(run):
    user_text = f"Question: {run['question']}\n\nContext:\n{run['context']}\n\nAnswer: {run['answer']}"
    out = _judge_chat_json('faceted', FACETED_JUDGE_SYSTEM, user_text)
    faithfulness = _clip01(out.get('faithfulness'))
    answer_relevance = _clip01(out.get('answer_relevance'))
    context_relevance = _clip01(out.get('context_relevance'))
    if run['answerable']:
        correct = (not is_abstention(run['answer'])) and faithfulness >= 0.5 and answer_relevance >= 0.5
    else:
        correct = is_abstention(run['answer'])   # the right answer to an unanswerable question is abstention
    return {'correct': correct, 'faithfulness': faithfulness,
            'answer_relevance': answer_relevance, 'context_relevance': context_relevance}


JUDGES = {'lexical': judge_lexical, 'single': judge_single, 'faceted': judge_faceted}


def judge_runs(runs):
    return {name: [judge(run) for run in runs] for name, judge in JUDGES.items()}


dev_verdicts = judge_runs(dev_runs)

print(f'Registered judges: {", ".join(JUDGES)}.')
print(f'Produced {len(dev_runs)} verdict(s) per judge over the development split.')
print(f'Live judge calls this run: {live_calls["judge"]} (rest served from the eval cache).')
print('All judges scored the identical Section 1 answers; none saw the labels.')


## Metrics in This Stage — What Each Number Means

Section 4 validates each judge against the labels, then scores the pipeline with the chosen judge and reads the gates off the held-out split. Judge selection keeps the simplest judge whose verdicts agree with the label-derived ground truth; the triad is added for the facets the labels cannot see.

| Metric | What it measures | How it is computed | What it signifies |
|---|---|---|---|
| Judge-label agreement | whether a judge's verdicts match the ground truth we can check | share of questions where the judge's correct/incorrect verdict matches the label-derived truth (unanswerable to abstain, answerable to a work-naming answer) | Judge validity gate — a judge below threshold is not trusted; higher is better. |
| Faithfulness | whether the answer is supported by its context | mean judge faithfulness over answered questions | Isolates generation drift; a release gate; higher is better. |
| Answer relevance | whether the answer addresses the question | mean judge answer relevance over answered questions | Catches fluent off-topic answers; a release gate; higher is better. |
| Context relevance | whether retrieval delivered on-topic evidence | mean judge context relevance over all questions | Isolates retrieval and query-handling faults; a release gate; higher is better. |
| Recall at k | whether the right works reached the context | share of an answerable question's expected works that appear in the top-k passages, averaged | Work-level retrieval quality; a release gate; higher is better. |
| Mean Reciprocal Rank (MRR) | how early the first relevant work appears | mean of one divided by the rank of the first passage from any expected work | Ranking quality; higher is better. |
| Abstention accuracy | whether the system refuses the unanswerable | share of unanswerable questions answered with the refusal sentinel | Hard safety gate carried from stage 08; must be 1.0. |
| Citation validity | whether every answered claim is traceable | share of answered questions whose bracketed markers all resolve to a provided passage | Hard grounding gate carried from stage 08; must be 1.0. |


## 4. Measure and Inspect

- **Two measurements, in order:** first **validate the judges** by agreement with the label-derived ground truth on the development split; then, using the chosen judge, **score the pipeline** on both splits and compute the retrieval metrics.
- **Judge validation.** For each judge we compare its correct/incorrect verdict to what the labels make checkable: an unanswerable question is "correct" only if the system abstained, and an answerable one is "correct" only if the answer names its expected works. The agreement fraction is the judge's trustworthiness; a judge that disagrees with these hard facts cannot be trusted on the softer facets.
- **Retrieval metrics are judge-free.** Recall at k and Mean Reciprocal Rank (MRR) are computed at the work level directly from the citations — did a passage from each expected work reach the top-k, and how early — so retrieval quality is measured deterministically regardless of the judge.
- **Hard gates are carried forward.** Abstention accuracy and citation validity are recomputed here exactly as in stages 08-09; they remain constraints, not scores, and any build that breaks one fails release outright.
- **A worked verdict.** We print the faceted judge's triad on a cross-play answer and on an out-of-corpus refusal, so the three facets are visible by eye alongside the numbers.
- **Honest limit:** six development and three held-out questions can only pre-screen a judge and sanity-check the gates; the thresholds are set conservatively and the labeled set must grow to 50-plus before these numbers certify quality. The structure, not the sample size, is the deliverable.


In [ ]:
def label_truth(run):
    # The only correctness facts the labels give us: abstain on unanswerable, name the works on answerable.
    if not run['answerable']:
        return is_abstention(run['answer'])
    return (not is_abstention(run['answer'])) and names_expected_works(run['answer'], run['expected_works'])


def judge_agreement(runs, verdicts):
    matches = sum(1 for run, verdict in zip(runs, verdicts) if bool(verdict['correct']) == label_truth(run))
    return matches / max(len(runs), 1)


judge_validation = {name: judge_agreement(dev_runs, verdicts) for name, verdicts in dev_verdicts.items()}

print('Judge validation (agreement with label-derived ground truth, development split):')
for name, agreement in judge_validation.items():
    cost = 0 if name == 'lexical' else len(dev_runs)
    print(f'  {name:<8} agreement={agreement:.0%}  (~{cost} judge call(s) per split)')


# ---- retrieval metrics at the work level, straight from the citations (judge-free) ----
def works_in_order(citations):
    seen, ordered = set(), []
    for citation in sorted(citations, key=lambda c: c['marker']):
        if citation['work'] not in seen:
            seen.add(citation['work'])
            ordered.append(citation['work'])
    return ordered


def recall_at_k(expected_works, citations, k):
    if not expected_works:
        return None
    retrieved = {c['work'] for c in citations if c['marker'] <= k}
    return sum(1 for w in expected_works if w in retrieved) / len(expected_works)


def reciprocal_rank(expected_works, citations):
    if not expected_works:
        return None
    for rank, work in enumerate(works_in_order(citations), start=1):
        if work in expected_works:
            return 1.0 / rank
    return 0.0


def citations_resolve(answer, citations):
    markers = [int(m) for m in _CITATION_PATTERN.findall(answer)]
    valid = {c['marker'] for c in citations}
    return bool(markers) and all(m in valid for m in markers)


def score_split(runs, judge_name):
    """Score one split with the chosen judge plus the judge-free retrieval and hard-constraint metrics."""
    judge = JUDGES[judge_name]
    verdicts = [judge(run) for run in runs]
    answerable = [(r, v) for r, v in zip(runs, verdicts) if r['answerable']]
    unanswerable = [(r, v) for r, v in zip(runs, verdicts) if not r['answerable']]
    answered = [(r, v) for r, v in answerable if not is_abstention(r['answer'])]
    faithful = [v['faithfulness'] for _, v in answered if v['faithfulness'] is not None]
    relevant = [v['answer_relevance'] for _, v in answered if v['answer_relevance'] is not None]
    context_rel = [v['context_relevance'] for _, v in zip(runs, verdicts) if v['context_relevance'] is not None]
    recalls = [recall_at_k(r['expected_works'], r['citations'], TOP_K) for r, _ in answerable]
    recalls = [x for x in recalls if x is not None]
    rrs = [reciprocal_rank(r['expected_works'], r['citations']) for r, _ in answerable]
    rrs = [x for x in rrs if x is not None]
    abstained_ok = sum(1 for r, _ in unanswerable if is_abstention(r['answer']))
    cite_ok = sum(1 for r, _ in answered if citations_resolve(r['answer'], r['citations']))
    return {
        'n': len(runs),
        'faithfulness': (sum(faithful) / len(faithful)) if faithful else 0.0,
        'answer_relevance': (sum(relevant) / len(relevant)) if relevant else 0.0,
        'context_relevance': (sum(context_rel) / len(context_rel)) if context_rel else 0.0,
        'recall_at_k': (sum(recalls) / len(recalls)) if recalls else 0.0,
        'mrr': (sum(rrs) / len(rrs)) if rrs else 0.0,
        'abstention_accuracy': abstained_ok / max(len(unanswerable), 1),
        'citation_validity': (cite_ok / len(answered)) if answered else 0.0,
        'answer_rate': len(answered) / max(len(answerable), 1),
    }


## 5. Justify the Selection

- **What is being selected:** not a pipeline component but the **release judge** and the **gates**. The judge must be trustworthy (agree with the labels) and must supply the facets the gates depend on; the gates are the thresholds a build clears on the held-out split to ship.
- **Judge decision rule (data-driven, deterministic):** keep only judges whose development-split agreement with the label-derived ground truth clears the validity bar. Among those, the release judge must also produce all three triad facets, because faithfulness and context relevance are gates the labels cannot measure — which only the faceted judge supplies. The deterministic lexical proxy is kept regardless, as the free pre-filter that runs on every commit before the paid judge.
- **Why the faceted judge, stated honestly:** the single judge compresses every failure into one bit and the lexical proxy is blind to paraphrase, so neither can power a faithfulness or context-relevance gate. The faceted judge can — provided its verdicts agree with the labels. If its agreement falls below the bar, the notebook flags it and falls back rather than trusting an uncalibrated judge.
- **Gate decision rule:** the two hard constraints inherited from stage 08 (perfect abstention, perfect citation validity) must hold exactly; the quality gates (faithfulness, answer relevance, context relevance, recall at k) must clear conservative thresholds. All gates are read on the **held-out** split, never the development split, so they estimate generalization rather than tuning.
- **Expected result:** the selected judge with its agreement, the dev and held-out scores, and a gate-by-gate promotion/rollback verdict for the current build.


In [ ]:
JUDGE_AGREEMENT_GATE = 0.80
SIMPLICITY_ORDER = {'lexical': 0, 'single': 1, 'faceted': 2}
REQUIRED_FACET_JUDGES = {'faceted'}   # only the faceted judge supplies faithfulness + context relevance


def provides_required_facets(name):
    return name in REQUIRED_FACET_JUDGES


eligible = [name for name, agreement in judge_validation.items()
            if agreement >= JUDGE_AGREEMENT_GATE - 1e-9 and provides_required_facets(name)]
if eligible:
    selected_judge = min(eligible, key=lambda n: SIMPLICITY_ORDER.get(n, 99))
else:
    # No facet-providing judge cleared validation: fall back to the best-agreeing judge and flag it.
    selected_judge = max(judge_validation, key=judge_validation.get)
    print('WARNING: no facet-providing judge cleared the agreement gate; falling back and flagging for review.')

CI_PREFILTER_JUDGE = 'lexical'   # kept as the free per-commit pre-filter regardless of the release judge
print(f'Selected release judge: {selected_judge} (agreement {judge_validation[selected_judge]:.0%}).')
print(f'Kept as per-commit pre-filter: {CI_PREFILTER_JUDGE} (free, deterministic).\n')

dev_scores = score_split(dev_runs, selected_judge)
heldout_scores = score_split(heldout_runs, selected_judge)

header = ('split', 'faith', 'ansrel', 'ctxrel', f'rec@{TOP_K}', 'mrr', 'abstain', 'cite', 'answer')
print('{:<9} {:>6} {:>7} {:>7} {:>7} {:>6} {:>8} {:>6} {:>7}'.format(*header))
for split_name, s in (('dev', dev_scores), ('held-out', heldout_scores)):
    print('{:<9} {:>6.0%} {:>7.0%} {:>7.0%} {:>7.0%} {:>6.2f} {:>8.0%} {:>6.0%} {:>7.0%}'.format(
        split_name, s['faithfulness'], s['answer_relevance'], s['context_relevance'],
        s['recall_at_k'], s['mrr'], s['abstention_accuracy'], s['citation_validity'], s['answer_rate']))

# ---- release gates, read on the held-out split ----
GATES = {
    'abstention_accuracy': ('==', 1.0), 'citation_validity': ('==', 1.0),
    'faithfulness': ('>=', 0.80), 'answer_relevance': ('>=', 0.80),
    'context_relevance': ('>=', 0.60), 'recall_at_k': ('>=', 0.80),
}


def gate_passes(operator, value, threshold):
    return abs(value - threshold) < 1e-9 if operator == '==' else value >= threshold - 1e-9


gate_results = {name: gate_passes(op, heldout_scores[name], thr) for name, (op, thr) in GATES.items()}
promote = all(gate_results.values())

print('\nRelease gates (held-out split):')
for name, (op, thr) in GATES.items():
    status = 'PASS' if gate_results[name] else 'FAIL'
    print(f'  {name:<20} {op} {thr:<5} -> {heldout_scores[name]:.2f}  [{status}]')
print(f'\nDecision: {"PROMOTE - all gates pass" if promote else "ROLLBACK - a gate failed; keep the last promoted build"}')

# Worked faceted verdicts, visible by eye.
cross = next((r for r in dev_runs if r['answerable'] and len(r['expected_works']) > 1), None)
refusal = next((r for r in dev_runs if not r['answerable']), None)
if cross:
    v = judge_faceted(cross)
    print(f'\nFaceted verdict - "{cross["question"][:60]}...": '
          f"faith={v['faithfulness']:.2f} ansrel={v['answer_relevance']:.2f} ctxrel={v['context_relevance']:.2f}")
if refusal:
    v = judge_faceted(refusal)
    print(f'Faceted verdict - out-of-corpus refusal: abstained={is_abstention(refusal["answer"])}, '
          f"ctxrel={v['context_relevance']:.2f}")


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/evaluation-results.json` — the standing evaluation contract stages 11-12 enforce: the selected release judge and its validated agreement, the kept lexical pre-filter, the development and held-out triad and retrieval scores, the per-category breakdown, the exact gate thresholds, and the current build's promotion/rollback decision.
- **Why a results-and-gates handoff:** stage 11 must not ship a build that fails these gates, and stage 12 monitors live traffic against the same thresholds. Exporting the judge, the thresholds, and the decision together makes "good enough to ship" a reproducible artifact rather than a conversation.
- **Production target:** `production/evaluation/dataset.py` loads the labeled questions and `production/evaluation/metrics.py` holds the judge-free retrieval metrics (recall at k, reciprocal rank) plus the hard-constraint checks, so the same scoring runs in Continuous Integration (CI) as ran here.
- **Honest limit, recorded in the handoff:** the labeled set is tiny, so the gates are provisional. The single most valuable next action is growing `eval-questions.json` to 50-plus questions across every category; the gate thresholds are written to be re-tightened as the set grows.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the handoff or the eval cache.
- **Expected result:** the handoff written, with the selected judge, the held-out scores, and the promotion/rollback decision printed.


In [ ]:
evaluation_results_path = ARTIFACTS / 'evaluation-results.json'


def category_breakdown(runs):
    by_category = defaultdict(list)
    for run in runs:
        by_category[run['category']].append(run)
    breakdown = {}
    for category, category_runs in by_category.items():
        verdicts = [JUDGES[selected_judge](run) for run in category_runs]
        answered = [(r, v) for r, v in zip(category_runs, verdicts)
                    if r['answerable'] and not is_abstention(r['answer'])]
        faith = [v['faithfulness'] for _, v in answered if v['faithfulness'] is not None]
        breakdown[category] = {
            'n': len(category_runs),
            'faithfulness': (sum(faith) / len(faith)) if faith else None,
            'agreement': judge_agreement(category_runs, [JUDGES[selected_judge](r) for r in category_runs]),
        }
    return breakdown


# evaluation-results.json: the standing judge + gates stages 11-12 enforce, plus the current build's verdict.
evaluation_results = {
    'stage': '10-continuous-evaluation',
    'release_judge': selected_judge,
    'judge_validation': judge_validation,
    'judge_agreement_gate': JUDGE_AGREEMENT_GATE,
    'ci_prefilter_judge': CI_PREFILTER_JUDGE,
    'selection_rule': 'judge must clear agreement gate and supply the RAG triad; gates read on held-out; lexical kept as free pre-filter',
    'scores': {'development': dev_scores, 'heldout': heldout_scores},
    'category_breakdown': {'development': category_breakdown(dev_runs), 'heldout': category_breakdown(heldout_runs)},
    'gates': {name: {'operator': op, 'threshold': thr, 'passed': gate_results[name]}
              for name, (op, thr) in GATES.items()},
    'decision': 'promote' if promote else 'rollback',
    'evaluated_pipeline': {
        'query_policy': SELECTED_POLICY, 'retrieval_method': SELECTED_METHOD, 'reranker': SELECTED_RERANKER,
        'context_builder': SELECTED_BUILDER, 'grounding_contract': SELECTED_CONTRACT, 'top_k': TOP_K,
    },
    'labeled_set_size': {'development': len(dev_runs), 'heldout': len(heldout_runs)},
    'production_target': {
        'modules': ['production/evaluation/dataset.py', 'production/evaluation/metrics.py'],
        'note': 'dataset.load_questions + metrics.recall_at_k/reciprocal_rank run the same scoring in CI',
    },
}
evaluation_results_path.write_text(json.dumps(evaluation_results, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote evaluation report to {evaluation_results_path.relative_to(ROOT)}')
print(f'Release judge: {selected_judge} | decision: {evaluation_results["decision"].upper()}')
print(f'Held-out faithfulness/answer/context: {heldout_scores["faithfulness"]:.2f} / '
      f'{heldout_scores["answer_relevance"]:.2f} / {heldout_scores["context_relevance"]:.2f}')
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit - do not commit the handoff.')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **The gates are only as strong as the labeled set.** With six development and three held-out questions, a single question moves a metric by a large fraction, so the thresholds are set conservatively and the handoff flags the set size as the top priority. The design is production-grade; the sample is not yet.
- **The judge is an instrument, and we calibrated the part we could.** We validated the judge against the two facts the labels give us — abstention and work naming — but faithfulness and context relevance are taken on the judge's word, unvalidated against human grades. Periodic human audits of the judge are the missing feedback loop.
- **The triad is where the real signal lives.** The hard constraints (abstention, citations) are pass/fail and the retrieval metrics are coarse at the work level; the triad is the only place that distinguishes a faithful answer from a fluent one. It is also the most expensive and the most judge-dependent — the part to watch as the set grows.
- **Held-out discipline is a habit, not a check.** Nothing in the code physically prevents tuning against the held-out split; only discipline does. The moment a future change is tuned until the held-out score improves, the honest estimate is gone. Keep tuning on development, read held-out once.
- **Caching makes evaluation reproducible and cheap.** Answers and judge verdicts are replayed from disk, so re-running compares against a fixed baseline and bills nothing. That is what lets evaluation run on every change rather than occasionally.
- **How we will revisit.** Grow the labeled set, re-validate the judge, re-tighten the gates, and add per-category minimums so a strong average cannot hide a broken category. The structure here is built to absorb all of that without redesign.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 04-09 artifacts, not hidden state. Query handling, embeddings, generations, and judge calls are all served from disk caches when warm, so a clean run bills nothing.
- **What we achieved:** the assembled pipeline run end to end over both labeled splits; three judges compared and validated against the label-derived ground truth; a selected release judge that supplies the Retrieval-Augmented Generation (RAG) triad, with the free lexical proxy kept as the per-commit pre-filter; development and held-out scores for faithfulness, answer relevance, context relevance, recall at k, and Mean Reciprocal Rank (MRR); the stage 08 hard constraints recomputed as gates; and an exported promotion/rollback decision.
- **Honest limits:** six development and three held-out questions can only pre-screen the judge and sanity-check the gates. The thresholds are conservative and provisional; growing `eval-questions.json` to 50-plus across every category is the highest-value next step, and the gates are written to tighten as it grows.
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the handoff or the eval cache.
- **Next stage — 11 Production and Serving:** load `.artifacts/evaluation-results.json`, compose the selected per-stage implementations into `production/pipeline.py`, wrap the online flow behind a tested FastAPI service with `/ask` and `/retrieve_evidence`, and choose operating settings (caching, concurrency) — refusing to ship any build that fails these gates.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* numbers. These questions build the transferable intuition.

**Q1. Why is evaluation a stage at all, instead of just reading the earlier metrics?**
Because every earlier metric scored one slice against a cheap proxy, and none of them answered the end-to-end question a user cares about: given a real question, is the final answer faithful, relevant, and built on relevant context? Assembling the whole pipeline and judging its answers is the first time those slices are tested *together*, and turning that into standing gates is what keeps the next change from silently breaking them.

**Q2. What is the Retrieval-Augmented Generation (RAG) triad really buying over one score?**
Localization. A single number tells you a build got worse; the triad tells you *where*. Low context relevance points upstream at retrieval or query handling — the evidence never arrived. High context relevance with low faithfulness points at the generator — the evidence arrived and was ignored. Fixing a regression starts with knowing which half of the pipeline to open, and the triad is what tells you.

**Q3. Why validate the judge instead of trusting it?**
An LLM judge is a model grading a model; it can agree with a confident, wrong answer. We only have a few hard facts the labels give us — which questions must be abstained and which works an answer must name — so we check the judge against exactly those, and only lean on it for faithfulness and relevance once it has proven it does not contradict the facts we can verify. A judge that fails that check is an unreliable instrument, no matter how fluent its reasons.

**Q4. Why read the gates on the held-out split and never tune against it?**
Every earlier stage tuned its choice on the development split, so a strong development score partly reflects that tuning rather than true quality. The held-out split was never used to choose anything, so it is the one honest estimate of how the system behaves on unseen questions. The moment you tune against it, it stops being held out and the estimate is gone.

**Q5. Why keep the lexical proxy after choosing a better judge?**
Cost and cadence. The faceted judge costs a model call per answer; the lexical proxy is free and instant. Running the free proxy on every commit catches blatant regressions in seconds, and the paid judge runs on the smaller set of changes that clear it. Cheap gate first, expensive gate second is standard CI economics.

**Q6. The sample is tiny — are these gates meaningful?**
As thresholds, provisionally; as *structure*, fully. Six development and three held-out questions cannot certify quality, and the handoff says so. But the machinery — a validated judge, the triad, work-level recall, hard constraints, and held-out gates — is exactly what a 500-question set would use. Growing the labeled set tightens the same gates without changing the design.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/evaluation-results.json` — the selected judge, the dev and held-out triad and retrieval scores, the gate thresholds, and the promotion/rollback decision. Stage 11 wires the gated pipeline behind a tested Application Programming Interface (API) and refuses to ship a build that fails these gates; stage 12 monitors live traffic against the same thresholds and drills the failure and rollback paths.
